# Feature Engineering - Tansaction Data - Growth Rate

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Transactions Monthly Ingest")
    # Parquet + AQE + Broadcast = speed
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.parquet.filterPushdown", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    .config("spark.sql.shuffle.partitions", "200")   # tune to cluster size
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '4g')
    .config('spark.executor.memory', '2g')
    .getOrCreate()
)


spark.conf.set("spark.sql.caseSensitive", "false")
spark


25/10/10 18:26:44 WARN Utils: Your hostname, I-Spider-Spacy-Y.local resolves to a loopback address: 127.0.0.1; using 10.13.115.158 instead (on interface en0)
25/10/10 18:26:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/10 18:26:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 62773)
Traceback (most recent call last):
  File "/opt/anaconda3/envs/tlc_env/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/anaconda3/envs/tlc_env/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/anaconda3/envs/tlc_env/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/anaconda3/envs/tlc_env/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/opt/anaconda3/envs/tlc_env/lib/python3.11/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/opt/anaconda3/envs/tlc_env/lib/python3.11/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
  

In [2]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pathlib import Path
from pyspark.sql.functions import col
from pyspark.sql.window import Window as W
from pyspark.sql.functions import row_number
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler


# Part 1: Load transaction
Load transactions across all three folders. 

And create features:
    ['merchant_abn', 'year_month', 'tx_count', 'unique_customers', 'total_dollar_value', 'avg_tx_value']

In [3]:
from pyspark.sql import functions as F
from functools import reduce

def process_parquet_folders_monthly(path):
    """
    Reads one or many parquet folders of daily-partitioned transactions
    and aggregates to merchant and month.

    Expected partition structure: order_datetime=YYYY-MM-DD
    Required fields inside parquet: merchant_abn, user_id, dollar_value
    """
    print(f"Reading {path} folder")

    # Load and union
    sdf_all = spark.read.parquet(path)

    # Basic filtering
    sdf_all = sdf_all.filter(
        (F.col("user_id") > 0) &
        (F.col("merchant_abn") > 0) &
        (F.col("dollar_value") > 0)
    )

    # Convert partition column to date
    # order_datetime will come in as string "YYYY-MM-DD"
    sdf_all = sdf_all.withColumn("order_date", F.to_date("order_datetime"))

    # Derive month bucket
    sdf_all = sdf_all.withColumn("month_start", F.date_trunc("month", F.col("order_date")))
    sdf_all = sdf_all.withColumn("year_month", F.date_format("month_start", "yyyy-MM"))

    # Aggregate
    monthly = (
        sdf_all.groupBy("merchant_abn", "year_month", "month_start")
        .agg(
            F.count(F.lit(1)).alias("tx_count"),
            F.countDistinct("user_id").alias("unique_customers"),
            F.sum("dollar_value").alias("total_dollar_value")
        )
        .withColumn("avg_tx_value",
            F.col("total_dollar_value") / F.col("tx_count")
        )
        .orderBy("merchant_abn", "month_start")
    )

    monthly = (
        monthly
            .filter(F.col("year_month") != "2021-02")            # 1) drop 2021-02
            .filter(F.col("year_month").between("2021-03", "2021-12"))
    )

    return monthly

path = "../data/raw/transactions_cleaned_parquet"


monthly_sdf = process_parquet_folders_monthly(path)
monthly_sdf.show(20, truncate=False)

Reading ../data/raw/transactions_cleaned_parquet folder


+------------+----------+-------------------+--------+----------------+------------------+------------------+
|merchant_abn|year_month|month_start        |tx_count|unique_customers|total_dollar_value|avg_tx_value      |
+------------+----------+-------------------+--------+----------------+------------------+------------------+
|10023283211 |2021-03   |2021-03-01 00:00:00|111     |111             |24634.345496105656|221.93104050545637|
|10023283211 |2021-04   |2021-04-01 00:00:00|129     |129             |27622.342754098943|214.1266880162709 |
|10023283211 |2021-05   |2021-05-01 00:00:00|144     |142             |30111.883260917697|209.11030042303958|
|10023283211 |2021-06   |2021-06-01 00:00:00|145     |145             |28790.434289032353|198.55471923470589|
|10023283211 |2021-07   |2021-07-01 00:00:00|153     |153             |29430.73191007144 |192.35772490242772|
|10023283211 |2021-08   |2021-08-01 00:00:00|146     |145             |32118.491546686848|219.9896681279921 |
|100232832

Given Data sturcture:
Part 1: consumer_fraud_probability.csv,
        consumer_user_details.parquet,
        merchant_fraud_probability.csv,
        tbl_consumer.csv,
        tbl_merchants.parquet.
Part 2: Transactions: 20210228 ~ 20210827
Part 3: Transactions: 20210828 ~ 20220227
Part 4: Transactions: 20220228 ~ 20221026

Transactions follows schema:
root
 |-- user_id: long (nullable = true)
 |-- merchant_abn: long (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)


order level records.

In [4]:
# Monthly merchant record shape
new_rows = monthly_sdf.count()
new_cols = len(monthly_sdf.columns)
print(f"Monthly merchant transaction record in 2021: {(new_rows, new_cols)}")

monthly_sdf.columns

Monthly merchant transaction record in 2021: (40566, 7)


['merchant_abn',
 'year_month',
 'month_start',
 'tx_count',
 'unique_customers',
 'total_dollar_value',
 'avg_tx_value']

In [5]:
monthly_sdf.printSchema()

root
 |-- merchant_abn: long (nullable = true)
 |-- year_month: string (nullable = true)
 |-- month_start: timestamp (nullable = true)
 |-- tx_count: long (nullable = false)
 |-- unique_customers: long (nullable = false)
 |-- total_dollar_value: double (nullable = true)
 |-- avg_tx_value: double (nullable = true)



# Part 2: Cleaning and Feature Engineering

In [6]:
# ==== Setup & cleanup ====
from pyspark.sql import functions as F, types as T, Window as W

# Treat IDs as strings to avoid Arrow/type issues
monthly_sdf = (
    monthly_sdf
    .withColumn("merchant_abn", F.col("merchant_abn").cast(T.StringType()))
)

# Ensure a month index per merchant for regression
w_idx = W.partitionBy("merchant_abn").orderBy("year_month")
monthly_idx = monthly_sdf.withColumn("month_idx", F.row_number().over(w_idx))

monthly_idx.show(5)

# Monthly merchant record shape (within 2021)
new_rows = monthly_sdf.count()
new_cols = len(monthly_sdf.columns)
print(f"Monthly merchant transaction record in 2021 (Mar - Dec): {(new_rows, new_cols)}")

monthly_sdf.columns

+------------+----------+-------------------+--------+----------------+------------------+------------------+---------+
|merchant_abn|year_month|        month_start|tx_count|unique_customers|total_dollar_value|      avg_tx_value|month_idx|
+------------+----------+-------------------+--------+----------------+------------------+------------------+---------+
| 10023283211|   2021-03|2021-03-01 00:00:00|     111|             111|24634.345496105656|221.93104050545637|        1|
| 10023283211|   2021-04|2021-04-01 00:00:00|     129|             129|27622.342754098943| 214.1266880162709|        2|
| 10023283211|   2021-05|2021-05-01 00:00:00|     144|             142|30111.883260917697|209.11030042303958|        3|
| 10023283211|   2021-06|2021-06-01 00:00:00|     145|             145|28790.434289032353|198.55471923470589|        4|
| 10023283211|   2021-07|2021-07-01 00:00:00|     153|             153| 29430.73191007144|192.35772490242772|        5|
+------------+----------+---------------

Monthly merchant transaction record in 2021 (Mar - Dec): (40566, 7)


['merchant_abn',
 'year_month',
 'month_start',
 'tx_count',
 'unique_customers',
 'total_dollar_value',
 'avg_tx_value']

In [7]:
# ==== 2) Per-merchant regression slopes ====
w = W.partitionBy("merchant_abn").orderBy("month_start")

# Prepare features for regression
assembler = VectorAssembler(inputCols=["month_idx"], outputCol="features")
df_features = assembler.transform(monthly_idx.withColumnRenamed("total_dollar_value", "label"))

# Monthly_sdf merge size
new_rows = monthly_sdf.count()
new_cols = len(monthly_sdf.columns)
print(f"Monthly merchant transaction record: {(new_rows, new_cols)}")

# Use groupBy + applyInPandas to run sklearn/MLlib per merchant
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("merchant_abn", StringType(), False),
    StructField("slope", DoubleType(), True),
    StructField("intercept", DoubleType(), True)
])

def fit_regression(pdf: pd.DataFrame) -> pd.DataFrame:
    from sklearn.linear_model import LinearRegression
    X = pdf[["month_idx"]].values
    y = pdf["total_dollar_value"].values
    model = LinearRegression().fit(X, y)
    return pd.DataFrame({
        "merchant_abn": [pdf["merchant_abn"].iloc[0]],
        "slope": [model.coef_[0]],
        "intercept": [model.intercept_]
    })

growth_sdf = monthly_idx.groupBy("merchant_abn").applyInPandas(fit_regression, schema)

Monthly merchant transaction record: (40566, 7)


In [8]:
# Join slope info back to the monthly_sdf
monthly_with_growth = (
    monthly_sdf.alias("m")
    .join(growth_sdf.alias("g"), on="merchant_abn", how="left")
)

w = W.partitionBy("merchant_abn").orderBy("month_start")
monthly_with_growth = monthly_with_growth.withColumn("month_idx", row_number().over(w))

monthly_with_growth.show(10)

+------------+----------+-------------------+--------+----------------+------------------+------------------+----------------+------------------+---------+
|merchant_abn|year_month|        month_start|tx_count|unique_customers|total_dollar_value|      avg_tx_value|           slope|         intercept|month_idx|
+------------+----------+-------------------+--------+----------------+------------------+------------------+----------------+------------------+---------+
| 10023283211|   2021-03|2021-03-01 00:00:00|     111|             111|24634.345496105656|221.93104050545637|2421.76669664197|20776.416869012035|        1|
| 10023283211|   2021-04|2021-04-01 00:00:00|     129|             129|27622.342754098943| 214.1266880162709|2421.76669664197|20776.416869012035|        2|
| 10023283211|   2021-05|2021-05-01 00:00:00|     144|             142|30111.883260917697|209.11030042303958|2421.76669664197|20776.416869012035|        3|
| 10023283211|   2021-06|2021-06-01 00:00:00|     145|          

In [9]:
def build_merchant_growth_features(monthly_df, tiny=1e-6):
    """
    Returns a per-merchant feature dataframe with:
      - std_slope              (recommended "normalised slope" = slope / mean level)
      - trend_nrmse            (RMSE / mean)
      - r2                     (share of variance explained by trend)
      - pos_month_share        (directional consistency vs trend)
      - inactivity_rate        (share of zero/near-zero months)
      - cagr_overall           (first non-zero -> last non-zero, annualised)
    """

    # Short alias to the value used for growth features
    df = monthly_df.withColumn("v", F.col("total_dollar_value"))

    # Windows
    w_ord = W.partitionBy("merchant_abn").orderBy("month_start")
    w_all = W.partitionBy("merchant_abn")
    w_ord_full = w_ord.rowsBetween(W.unboundedPreceding, W.unboundedFollowing)

    # ----- Fitted values & residuals -----
    df = df.withColumn("y_hat", F.col("intercept") + F.col("slope") * F.col("month_idx"))
    df = df.withColumn("resid", F.col("v") - F.col("y_hat"))

    # ----- Inactivity flag -----
    df = df.withColumn("inactive_flag", F.when(F.col("v") <= F.lit(tiny), 1).otherwise(0))

    # ----- Overall CAGR (first & last non-zero months) -----
    df = df.withColumn("v_nz", F.when(F.col("v") > F.lit(tiny), F.col("v")))
    df = df.withColumn("first_nz_v",     F.first("v_nz", ignorenulls=True).over(w_ord_full))
    df = df.withColumn("first_nz_month", F.first(F.when(F.col("v") > F.lit(tiny), F.col("month_start")), ignorenulls=True).over(w_ord_full))
    df = df.withColumn("last_nz_v",      F.last("v_nz", ignorenulls=True).over(w_ord_full))
    df = df.withColumn("last_nz_month",  F.last(F.when(F.col("v") > F.lit(tiny), F.col("month_start")), ignorenulls=True).over(w_ord_full))

    df = df.withColumn("months_between",
                       F.months_between(F.col("last_nz_month"), F.col("first_nz_month")))
    df = df.withColumn(
        "cagr_overall_tmp",
        F.when((F.col("first_nz_v") > F.lit(tiny)) & (F.col("last_nz_v").isNotNull()) & (F.col("months_between") > 0),
               F.pow(F.col("last_nz_v") / F.col("first_nz_v"),
                     12.0 / F.col("months_between")) - 1
        )
    )

    return df

metrics_sdf = build_merchant_growth_features(monthly_with_growth, tiny=1e-6)
metrics_sdf.show(5)

+------------+----------+-------------------+--------+----------------+------------------+------------------+----------------+------------------+---------+------------------+------------------+-------------------+-------------+------------------+------------------+-------------------+-----------------+-------------------+--------------+------------------+
|merchant_abn|year_month|        month_start|tx_count|unique_customers|total_dollar_value|      avg_tx_value|           slope|         intercept|month_idx|                 v|             y_hat|              resid|inactive_flag|              v_nz|        first_nz_v|     first_nz_month|        last_nz_v|      last_nz_month|months_between|  cagr_overall_tmp|
+------------+----------+-------------------+--------+----------------+------------------+------------------+----------------+------------------+---------+------------------+------------------+-------------------+-------------+------------------+------------------+-------------------

In [10]:
metrics_sdf.printSchema()

root
 |-- merchant_abn: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- month_start: timestamp (nullable = true)
 |-- tx_count: long (nullable = false)
 |-- unique_customers: long (nullable = false)
 |-- total_dollar_value: double (nullable = true)
 |-- avg_tx_value: double (nullable = true)
 |-- slope: double (nullable = true)
 |-- intercept: double (nullable = true)
 |-- month_idx: integer (nullable = false)
 |-- v: double (nullable = true)
 |-- y_hat: double (nullable = true)
 |-- resid: double (nullable = true)
 |-- inactive_flag: integer (nullable = false)
 |-- v_nz: double (nullable = true)
 |-- first_nz_v: double (nullable = true)
 |-- first_nz_month: timestamp (nullable = true)
 |-- last_nz_v: double (nullable = true)
 |-- last_nz_month: timestamp (nullable = true)
 |-- months_between: double (nullable = true)
 |-- cagr_overall_tmp: double (nullable = true)



In [11]:
from pyspark.sql import functions as F

# Small epsilon to prevent divide-by-zero
tiny = 1e-6  

metrics_agg_sdf = (
    metrics_sdf.groupBy("merchant_abn")
    .agg(
        # Standardised slope = slope / avg_y
        (F.first("slope") / (F.avg("y_hat") + F.lit(tiny))).alias("std_slope"),

        # Trend error (normalised RMSE)
        (F.sqrt(F.avg(F.col("resid")**2)) / (F.avg("y_hat") + F.lit(tiny))).alias("trend_nrmse"),

        # R² = 1 - SSE/SST
        (1 - (F.sum(F.col("resid")**2) / (F.sum(F.col("v") + tiny)))).alias("r2"),

        # Positive month share = share of months with y > 0
        (F.sum(F.when(F.col("y_hat") > 0, 1).otherwise(0)) / F.count("*")).alias("pos_month_share"),

        # Inactivity rate = average of inactive_flag
        F.avg("inactive_flag").alias("inactivity_rate"),

        # CAGR (take last value of precomputed cagr_overall_tmp)
        F.max("cagr_overall_tmp").alias("cagr_overall"),
    )
)

metrics_agg_sdf.show(10, truncate=False)


+------------+--------------------+-------------------+-------------------+---------------+---------------+------------------+
|merchant_abn|std_slope           |trend_nrmse        |r2                 |pos_month_share|inactivity_rate|cagr_overall      |
+------------+--------------------+-------------------+-------------------+---------------+---------------+------------------+
|10023283211 |0.07102760441522976 |0.08217225152747205|-229.22660488373512|1.0            |0.0            |1.2375973355450736|
|10142254217 |0.0515848915953296  |0.15246445544494838|-133.53563400526915|1.0            |0.0            |0.8995355010890165|
|10165489824 |0.0                 |0.0                |1.0                |1.0            |0.0            |NULL              |
|10187291046 |0.030933493121289107|0.21672476424486942|-86.69916313434048 |1.0            |0.0            |0.5720700960321818|
|10192359162 |0.04842268626514383 |0.31102358630178667|-818.7166524068284 |1.0            |0.0            |1.48

In [12]:
# ==== 3) Per-merchant aggregates & “latest month” snapshot ====
# Latest month per merchant
w_latest = W.partitionBy("merchant_abn").orderBy(F.col("year_month").desc())
latest = (
    monthly_idx
      .withColumn("rn", F.row_number().over(w_latest))
      .filter(F.col("rn") == 1)
      .select(
          "merchant_abn", 
          F.col("year_month").alias("latest_year_month"),
          F.col("tx_count").alias("latest_tx_count"),
          F.col("unique_customers").alias("latest_unique_customers"),
          F.col("total_dollar_value").alias("latest_total_dollar_value"),
          F.col("avg_tx_value").alias("latest_avg_tx_value")
      )
)


# Aggregates across all available months
agg_sdf = (
    monthly_idx.groupBy("merchant_abn")
    .agg(
        F.count("*").alias("active_months"),
        F.min("year_month").alias("first_year_month"),
        F.max("year_month").alias("last_year_month"),

        F.sum("tx_count").alias("sum_tx_count"),
        F.avg("tx_count").alias("avg_monthly_tx_count"),
        

        F.sum("total_dollar_value").alias("sum_total_dollar_value"),
        F.avg("total_dollar_value").alias("avg_monthly_total_dollar_value"),
        
        F.avg("unique_customers").alias("avg_monthly_unique_customers"),
        # F.avg("avg_tx_value").alias("avg_of_monthly_avg_tx_value"),
    )
    # overall average ticket value = total revenue / total tx (more stable than avg of monthly avgs)
    .withColumn(
        "overall_avg_tx_value",
        F.when(F.col("sum_tx_count") > 0, F.col("sum_total_dollar_value") / F.col("sum_tx_count")).otherwise(F.lit(None))
    )
)

In [13]:
agg_sdf.show(5)

+------------+-------------+----------------+---------------+------------+--------------------+----------------------+------------------------------+----------------------------+--------------------+
|merchant_abn|active_months|first_year_month|last_year_month|sum_tx_count|avg_monthly_tx_count|sum_total_dollar_value|avg_monthly_total_dollar_value|avg_monthly_unique_customers|overall_avg_tx_value|
+------------+-------------+----------------+---------------+------------+--------------------+----------------------+------------------------------+----------------------------+--------------------+
| 10023283211|           10|         2021-03|        2021-12|        1600|               160.0|    340961.33700542874|             34096.13370054287|                       159.4|  213.10083562839296|
| 10142254217|           10|         2021-03|        2021-12|        1492|               149.2|     57876.21426090408|             5787.621426090408|                       148.6|  38.791028325002735|


In [14]:
# ==== 4) Combine everything into the overall merchant table ====
merchant_overall_sdf = (
    agg_sdf.alias("a")
    .join(metrics_agg_sdf.alias("s"), on="merchant_abn", how="left")
)

# Optional: order columns
select_cols = [
    "merchant_abn",
    "active_months", "first_year_month", "last_year_month",

    "sum_tx_count", "avg_monthly_tx_count", "overall_avg_tx_value",
     
    "sum_total_dollar_value", "avg_monthly_total_dollar_value",
    
    "avg_monthly_unique_customers",

    "std_slope", "trend_nrmse", "r2", "pos_month_share", "inactivity_rate","cagr_overall"

    # "latest_year_month"
    
    # "latest_tx_count", "latest_unique_customers", "latest_total_dollar_value", "latest_avg_tx_value",
    # "slope_tx_count", "slope_unique_customers", "slope_total_dollar_value", "slope_avg_tx_value",
]
merchant_overall_sdf = merchant_overall_sdf.select(*select_cols)



In [15]:
merchant_overall_sdf.show(10)

+------------+-------------+----------------+---------------+------------+--------------------+--------------------+----------------------+------------------------------+----------------------------+--------------------+-------------------+-------------------+---------------+---------------+------------------+
|merchant_abn|active_months|first_year_month|last_year_month|sum_tx_count|avg_monthly_tx_count|overall_avg_tx_value|sum_total_dollar_value|avg_monthly_total_dollar_value|avg_monthly_unique_customers|           std_slope|        trend_nrmse|                 r2|pos_month_share|inactivity_rate|      cagr_overall|
+------------+-------------+----------------+---------------+------------+--------------------+--------------------+----------------------+------------------------------+----------------------------+--------------------+-------------------+-------------------+---------------+---------------+------------------+
| 10023283211|           10|         2021-03|        2021-12|   

In [16]:
# Monthly_sdf merge size
new_rows = merchant_overall_sdf.count()
new_cols = len(merchant_overall_sdf.columns)
print(f"Merchant transaction record: {(new_rows, new_cols)}")

Merchant transaction record: (4406, 16)


In [17]:
# Get the overall unique user count per merchant 
def process_parquet_folder(path):
    """
    Reads a parquet folder, prints size before and after filtering,
    and returns the filtered Spark DataFrame.
    
    Args:
        path (str): Folder path to read parquet files from
    
    Returns:
        Spark DataFrame: filtered dataframe
    """
    print(f"\nProcessing folder: {path}")
    
    # Step 1: Read the parquet into sdf
    sdf = spark.read.parquet(path)
    
    # Step 2: Original size
    num_rows = sdf.count()
    num_cols = len(sdf.columns)
    print(f"Original size: {(num_rows, num_cols)}")
    
    # Step 3: Add validity column
    sdf = sdf.withColumn(
        "is_valid_record",
        F.when(
            (F.col("user_id") > 0) &
            (F.col("merchant_abn") > 0) &
            (F.col("dollar_value") > 0),
            True
        ).otherwise(False)
    )
    
    # Step 4: Filter valid records
    sdf = sdf.filter(F.col("is_valid_record") == True)
    
    # Step 5: Filtered size
    new_rows = sdf.count()
    new_cols = len(sdf.columns)
    print(f"--- Filtered size: {(new_rows, new_cols)}")
    
    return sdf


In [18]:
# input paths
P2_path = "../data/given/part 2/tables/transactions_20210228_20210827_snapshot/"
P3_path = "../data/given/part 3/tables/transactions_20210828_20220227_snapshot/"
P4_path = "../data/given/part 4/tables/transactions_20220228_20220828_snapshot/"\


p2_sdf = process_parquet_folder(P2_path)
p3_sdf = process_parquet_folder(P3_path)
p4_sdf = process_parquet_folder(P4_path)


Processing folder: ../data/given/part 2/tables/transactions_20210228_20210827_snapshot/
Original size: (3643266, 5)
--- Filtered size: (3643266, 6)

Processing folder: ../data/given/part 3/tables/transactions_20210828_20220227_snapshot/
Original size: (4508106, 5)
--- Filtered size: (4508106, 6)

Processing folder: ../data/given/part 4/tables/transactions_20220228_20220828_snapshot/
Original size: (6044133, 5)
--- Filtered size: (6044133, 6)


In [19]:
# 1) Union the three period DFs (schema-safe)
all_sdf = (
    p2_sdf.unionByName(p3_sdf, allowMissingColumns=True)
          .unionByName(p4_sdf, allowMissingColumns=True)
)

# Convert partition column to date
# order_datetime will come in as string "YYYY-MM-DD"
all_sdf = all_sdf.withColumn("order_date", F.to_date("order_datetime"))

# Derive month bucket
all_sdf = all_sdf.withColumn("month_start", F.date_trunc("month", F.col("order_date")))
all_sdf = all_sdf.withColumn("year_month", F.date_format("month_start", "yyyy-MM"))



# 2) (Optional) keep only the columns we need and basic sanity filter
all_sdf = (
    all_sdf.filter((F.col("year_month").between("2021-03", "2021-12"))
))

# 3) Unique users per merchant across the whole time range
unique_users_df = (
    all_sdf.groupBy("merchant_abn")
           .agg(
               F.countDistinct("user_id").alias("unique_users_all_periods"),
           )
           .orderBy("merchant_abn")
)

# Inspect
unique_users_df.show(20, truncate=False)

+------------+------------------------+
|merchant_abn|unique_users_all_periods|
+------------+------------------------+
|10023283211 |1545                    |
|10142254217 |1433                    |
|10165489824 |1                       |
|10187291046 |165                     |
|10192359162 |183                     |
|10206519221 |4357                    |
|10255988167 |422                     |
|10264435225 |2331                    |
|10279061213 |270                     |
|10323485998 |4625                    |
|10342410215 |424                     |
|10346855916 |6                       |
|10364012396 |14                      |
|10385011947 |15                      |
|10385163239 |33                      |
|10385250025 |340                     |
|10404542215 |1                       |
|10430380319 |72                      |
|10441711491 |1                       |
|10462560289 |788                     |
+------------+------------------------+
only showing top 20 rows



In [20]:
merchant_overall_sdf = (
    merchant_overall_sdf.alias("a")
    .join(unique_users_df.alias("s"), on="merchant_abn", how="left")
)

merchant_overall_sdf.show(10)

+------------+-------------+----------------+---------------+------------+--------------------+--------------------+----------------------+------------------------------+----------------------------+--------------------+-------------------+-------------------+---------------+---------------+------------------+------------------------+
|merchant_abn|active_months|first_year_month|last_year_month|sum_tx_count|avg_monthly_tx_count|overall_avg_tx_value|sum_total_dollar_value|avg_monthly_total_dollar_value|avg_monthly_unique_customers|           std_slope|        trend_nrmse|                 r2|pos_month_share|inactivity_rate|      cagr_overall|unique_users_all_periods|
+------------+-------------+----------------+---------------+------------+--------------------+--------------------+----------------------+------------------------------+----------------------------+--------------------+-------------------+-------------------+---------------+---------------+------------------+---------------

In [21]:
merchant_overall_sdf.printSchema()

root
 |-- merchant_abn: string (nullable = true)
 |-- active_months: long (nullable = false)
 |-- first_year_month: string (nullable = true)
 |-- last_year_month: string (nullable = true)
 |-- sum_tx_count: long (nullable = true)
 |-- avg_monthly_tx_count: double (nullable = true)
 |-- overall_avg_tx_value: double (nullable = true)
 |-- sum_total_dollar_value: double (nullable = true)
 |-- avg_monthly_total_dollar_value: double (nullable = true)
 |-- avg_monthly_unique_customers: double (nullable = true)
 |-- std_slope: double (nullable = true)
 |-- trend_nrmse: double (nullable = true)
 |-- r2: double (nullable = true)
 |-- pos_month_share: double (nullable = true)
 |-- inactivity_rate: double (nullable = true)
 |-- cagr_overall: double (nullable = true)
 |-- unique_users_all_periods: long (nullable = true)



In [22]:
# ==== 6) Customer level Features ====
from pyspark.sql.functions import broadcast

path = "../data/raw/transactions_cleaned_parquet"

# 1) Read whole folder (Spark will read all parquet files under the directory)
transactions = spark.read.parquet(path)

# Linking this to ABS census data by postcode (2021 Census)
consumer_path = "../data/curated/consumer_curated_with_postcode.csv"
consumer_sdf = (spark.read
                .option("header", True)
                .option("inferSchema", True)
                .csv(consumer_path)
                .withColumn("postcode", F.trim(F.col("postcode").cast("string"))))


joined_sdf = transactions.join(broadcast(consumer_sdf), on="user_id", how="left")

# Quick inspection
print("transactions rows:", transactions.count(), "consumer rows:", consumer_sdf.count(), "joined rows:", joined_sdf.count())
joined_sdf.show(5, truncate=False)

# Removing records with empty name and null total population in the postcode area 
sdf = joined_sdf.filter(F.col("Tot_P_P").isNotNull() & (F.col("name") != ""))

25/10/10 18:28:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


transactions rows: 12561377 consumer rows: 491941 joined rows: 12561377


+-------+------------+------------------+------------------------------------+------------+-------------------+----------+-----------+---------------+------------------------------+-----+--------+-----------+-----------+---------------------+---------------------+---------------------+-------+----------------------------+-----------------------------+---------------------------+------------------+-------------------------+----------------------------+-------------------------+------------------+----------------+------------+---------------+------------+
|user_id|merchant_abn|dollar_value      |order_id                            |source_batch|order_datetime     |Unnamed: 0|consumer_id|name           |address                       |state|postcode|gender     |fraud_count|avg_fraud_probability|max_fraud_probability|min_fraud_probability|Tot_P_P|High_yr_schl_comp_Yr_12_eq_P|Median_mortgage_repay_monthly|Median_tot_prsnl_inc_weekly|Median_rent_weekly|Median_tot_fam_inc_weekly|Average_num_ps

In [23]:
# Keep merchant_abn type consistent with merchant_overall_sdf (string)
merchant_overall_sdf = merchant_overall_sdf.withColumn("merchant_abn", F.col("merchant_abn").cast(T.StringType()))
joined_sdf          = joined_sdf.withColumn("merchant_abn", F.col("merchant_abn").cast(T.StringType()))


In [24]:
# Scope if needed (e.g., 2021 only) — comment out if not required
# scoped = joined_sdf.filter(F.to_date("order_datetime").between("2021-03-01","2021-12-31"))
scoped = joined_sdf.select("merchant_abn", "user_id", "order_id", "order_datetime")

# Deduplicate exact same tx (protects against double loads)
scoped = scoped.dropDuplicates(["merchant_abn", "user_id", "order_id"])

# Repeat customers (≥2 tx per merchant)
repeat_per_merchant = (
    scoped.groupBy("merchant_abn", "user_id").agg(F.count("*").alias("txs"))
          .filter(F.col("txs") >= 2)
          .groupBy("merchant_abn")
          .agg(F.countDistinct("user_id").alias("repeat_customers"))
)

# Unique customers per merchant (recomputed; avoids ambiguity)
unique_customer = (
    scoped.select("merchant_abn", "user_id").distinct()
          .groupBy("merchant_abn")
          .agg(F.count("*").alias("unique_users_all_periods"))
)

# Join uniques + repeats, then merge into your overall table
repeat_stats = (
    unique_customer.alias("u")
    .join(repeat_per_merchant.alias("r"), on="merchant_abn", how="left")
    .fillna({"repeat_customers": 0})
    .withColumn("repeat_customers", F.col("repeat_customers").cast("long"))
    .withColumn("repeat_share", (F.col("repeat_customers") / F.col("u.unique_users_all_periods")).cast("double"))
)

merchant_overall_sdf = (
    merchant_overall_sdf.alias("m")
    .drop("unique_users_all_periods")  # drop the ambiguous copy/copies
    .join(repeat_stats.alias("rs"), on="merchant_abn", how="left")
)

merchant_overall_sdf.printSchema()

root
 |-- merchant_abn: string (nullable = true)
 |-- active_months: long (nullable = false)
 |-- first_year_month: string (nullable = true)
 |-- last_year_month: string (nullable = true)
 |-- sum_tx_count: long (nullable = true)
 |-- avg_monthly_tx_count: double (nullable = true)
 |-- overall_avg_tx_value: double (nullable = true)
 |-- sum_total_dollar_value: double (nullable = true)
 |-- avg_monthly_total_dollar_value: double (nullable = true)
 |-- avg_monthly_unique_customers: double (nullable = true)
 |-- std_slope: double (nullable = true)
 |-- trend_nrmse: double (nullable = true)
 |-- r2: double (nullable = true)
 |-- pos_month_share: double (nullable = true)
 |-- inactivity_rate: double (nullable = true)
 |-- cagr_overall: double (nullable = true)
 |-- unique_users_all_periods: long (nullable = true)
 |-- repeat_customers: long (nullable = true)
 |-- repeat_share: double (nullable = true)



In [25]:

# 3) Run Spark groupBy/aggregations without collecting to driver
demog_agg = (
    sdf.groupBy("merchant_abn")
    .agg(
        F.avg("Tert_Tot_Tert_P").alias("total_tertiary_ratio"),
        F.avg("High_yr_schl_comp_Yr_12_eq_P").alias("total_yr12_completion_ratio"),
        F.avg("Median_tot_prsnl_inc_weekly").alias("median_tot_prsnl_inc_weekly"),
        (F.count("*") / F.countDistinct("user_id")).alias("tx_per_customer")
        
    )
)

demog_agg.printSchema()

root
 |-- merchant_abn: long (nullable = true)
 |-- total_tertiary_ratio: double (nullable = true)
 |-- total_yr12_completion_ratio: double (nullable = true)
 |-- median_tot_prsnl_inc_weekly: double (nullable = true)
 |-- tx_per_customer: double (nullable = true)



In [26]:
# 4) Aggeragate the repeat customer and external features


merchant_overall_sdf = (
    merchant_overall_sdf.alias("a")
    .join(demog_agg.alias("s"), on="merchant_abn", how="left")
)

merchant_overall_sdf.printSchema()

root
 |-- merchant_abn: string (nullable = true)
 |-- active_months: long (nullable = false)
 |-- first_year_month: string (nullable = true)
 |-- last_year_month: string (nullable = true)
 |-- sum_tx_count: long (nullable = true)
 |-- avg_monthly_tx_count: double (nullable = true)
 |-- overall_avg_tx_value: double (nullable = true)
 |-- sum_total_dollar_value: double (nullable = true)
 |-- avg_monthly_total_dollar_value: double (nullable = true)
 |-- avg_monthly_unique_customers: double (nullable = true)
 |-- std_slope: double (nullable = true)
 |-- trend_nrmse: double (nullable = true)
 |-- r2: double (nullable = true)
 |-- pos_month_share: double (nullable = true)
 |-- inactivity_rate: double (nullable = true)
 |-- cagr_overall: double (nullable = true)
 |-- unique_users_all_periods: long (nullable = true)
 |-- repeat_customers: long (nullable = true)
 |-- repeat_share: double (nullable = true)
 |-- total_tertiary_ratio: double (nullable = true)
 |-- total_yr12_completion_ratio: doub

In [27]:
# ==== 6) Save result ====
# Parquet
import os
os.makedirs("../result", exist_ok=True)

merchant_feature_engineering = merchant_overall_sdf.toPandas()
merchant_feature_engineering.to_csv("../data/curated/merchant_feature_engineering.csv")


25/10/10 18:28:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/10 18:28:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/10 18:28:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/10 18:28:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/10 18:28:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/10 18:28:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/10 18:28:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/10 18:28:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/10/10 18:28:46 WARN RowBasedKeyValueBatch: Calling spill() on

In [28]:
merchant_feature_df = pd.read_csv('../data/curated/merchant_feature_engineering.csv')


In [29]:
merchant_feature_df.columns

Index(['Unnamed: 0', 'merchant_abn', 'active_months', 'first_year_month',
       'last_year_month', 'sum_tx_count', 'avg_monthly_tx_count',
       'overall_avg_tx_value', 'sum_total_dollar_value',
       'avg_monthly_total_dollar_value', 'avg_monthly_unique_customers',
       'std_slope', 'trend_nrmse', 'r2', 'pos_month_share', 'inactivity_rate',
       'cagr_overall', 'unique_users_all_periods', 'repeat_customers',
       'repeat_share', 'total_tertiary_ratio', 'total_yr12_completion_ratio',
       'median_tot_prsnl_inc_weekly', 'tx_per_customer'],
      dtype='object')

In [30]:
merchant_feature_df.head(5)

,Unnamed: 0,merchant_abn,active_months,first_year_month,last_year_month,sum_tx_count,avg_monthly_tx_count,overall_avg_tx_value,sum_total_dollar_value,avg_monthly_total_dollar_value,...,pos_month_share,inactivity_rate,cagr_overall,unique_users_all_periods,repeat_customers,repeat_share,total_tertiary_ratio,total_yr12_completion_ratio,median_tot_prsnl_inc_weekly,tx_per_customer
0,0,10142254217,10,2021-03,2021-12,1492,149.2,38.791028,57876.214261,5787.621426,...,1.0,0.0,0.899536,2510,148,0.058964,5.201302,40.076130,792.818182,1.061076
1,1,10187291046,10,2021-03,2021-12,165,16.5,113.160388,18671.463953,1867.146395,...,1.0,0.0,0.572070,296,1,0.003378,4.763940,39.364555,791.125000,1.003922
2,2,10192359162,10,2021-03,2021-12,183,18.3,463.048006,84737.785125,8473.778512,...,1.0,0.0,1.485803,329,1,0.003040,5.141724,40.423854,818.410256,1.003676
3,3,10206519221,10,2021-03,2021-12,4842,484.2,38.125672,184604.501867,18460.450187,...,1.0,0.0,1.236993,7118,1219,0.171256,5.134904,39.698740,790.694284,1.192560
4,4,10255988167,10,2021-03,2021-12,428,42.8,398.225257,170440.410057,17044.041006,...,1.0,0.0,1.387497,719,14,0.019471,5.027784,39.064928,792.150502,1.018739


25/10/10 21:32:36 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 577004 ms exceeds timeout 120000 ms
25/10/10 21:32:36 WARN SparkContext: Killing executors is not supported by current scheduler.
25/10/10 21:53:39 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o